# Train Nôm glyph classifier (Kaggle GPU)

Embedding-model phân biệt chữ Nôm (ResNet-18 + ArcFace) thay DINOv2 cho S3.

**Cách dùng:** đóng gói ở máy (`prepare_data.py` → `pack_for_kaggle.py`) → upload `kaggle_pkg/` làm **Kaggle Dataset** (DATA) → Add Input vào notebook này.
Code được **clone từ GitHub** (như `diffusion_run.ipynb`); DATA lấy từ dataset.

⚠️ **GPU:** PyTorch mặc định của Kaggle KHÔNG hỗ trợ **P100 (sm_60)**. **Khuyến nghị dùng `GPU T4 x2`** (chạy ngay). Muốn P100 thì chạy cell cài lại torch (1b).

💾 **Checkpoint không mất khi restart:** đặt `HF_TOKEN` (Secrets) + `HF_REPO` → push HF mỗi epoch + tự resume. Hoặc chạy bằng **Save & Run All (Commit)** để lưu Output.

## 1. Kiểm GPU

In [ ]:
import torch, shutil
assert shutil.which('nvidia-smi'), '🛑 Bật GPU: Settings → Accelerator'
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
ok   = torch.cuda.is_available() and cap >= (7,0)   # torch Kaggle cần sm_70+
print(f'GPU: {name} | capability sm_{cap[0]}{cap[1]} | torch {torch.__version__}')
if not ok:
    print('⚠️ GPU/torch KHÔNG tương thích (P100=sm_60).')
    print('   -> Đổi sang GPU T4 x2 (Settings) RỒI Restart, HOẶC chạy cell 1b để cài torch cu121 cho P100.')
else:
    print('✓ GPU tương thích, train được.')

## 1b. (CHỈ khi cố dùng P100) Cài lại torch hỗ trợ sm_60
Bỏ qua nếu đã chọn T4. Cài ~2–3 phút; sau đó **Restart session** rồi chạy lại từ cell 1.

In [ ]:
# Bỏ dấu # để chạy khi dùng P100:
# !pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
print('Nếu cần P100: bỏ comment dòng trên, chạy, rồi Run → Restart session.')

## 2. Clone code từ GitHub (như diffusion_run)
Lấy code mới nhất; nếu repo chưa có code thì fallback dùng code bundled trong DATA dataset.
*(Muốn clone lấy code mới: push `evaluation/ver_new/nom_classifier/` lên GitHub trước.)*

In [ ]:
import os, sys, glob
from pathlib import Path
REPO_URL='https://github.com/truong571/GanNhanOCR.git'
CLONE=Path('/kaggle/working/GanNhanOCR')
if not CLONE.exists():
    !git clone --depth 1 {REPO_URL} {CLONE}
else:
    !cd {CLONE} && git pull --ff-only || true
CODE = CLONE/'evaluation/ver_new/nom_classifier'
if not (CODE/'train.py').exists():
    # fallback: code bundled trong DATA dataset
    d=[os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.py', recursive=True)]
    CODE=Path(d[0]) if d else CODE
assert (CODE/'train.py').exists(), f'Không thấy train.py ở {CODE} — push code lên GitHub hoặc để code trong DATA pkg'
print('CODE =', CODE)

## 3. Trỏ DATA + (tuỳ chọn) HF token để lưu bền

In [ ]:
import os, glob
print('Datasets:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'TRỐNG — chưa Add Input')
hits = glob.glob('/kaggle/input/**/index.csv', recursive=True)   # recursive: bắt cả thư mục con
assert hits, 'Chưa thấy index.csv → Add Input dataset (panel phải).'
DATA = os.path.dirname(hits[0])
print('DATA =', DATA, '| images/crop:', os.path.isdir(f'{DATA}/images/crop'), '| classes.json:', os.path.exists(f'{DATA}/classes.json'))

## 3b. (Tuỳ chọn) Lưu checkpoint lên HF Hub — KHÔNG mất khi restart

**Bật:** Add-ons → Secrets → thêm `HF_TOKEN` = token role **Write** (https://huggingface.co/settings/tokens) → **Attach**. Xong — repo tự đặt `<username>/nom-embed`.
Không thêm secret = HF OFF (giữ checkpoint bằng Save & Run All).

In [ ]:
import os
HF_REPO = ''   # để trống = tự dùng '<username-HF>/nom-embed'. Sửa nếu muốn tên repo khác.
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception: pass
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import HfApi, create_repo
        who = HfApi().whoami(token=os.environ['HF_TOKEN'])['name']
        HF_REPO = HF_REPO or f'{who}/nom-embed'
        create_repo(HF_REPO, repo_type='model', exist_ok=True, token=os.environ['HF_TOKEN'])
        print(f'✓ HF ON: {HF_REPO} (user {who}) — push mỗi epoch + tự resume khi chạy lại')
    except Exception as e:
        print(f'✗ HF lỗi: {type(e).__name__}: {e}')
        print('  -> token có role=Write? đã Attach secret HF_TOKEN? (bỏ qua = train không HF)')
        HF_REPO = ''
else:
    print('HF OFF (chưa có secret HF_TOKEN) -> dùng Save & Run All (Commit) để giữ checkpoint')

## 4. Train
`--resume` tự tiếp tục từ `last.pt` (local hoặc kéo từ HF). `--hf-repo` push mỗi epoch.
Giảm `--batch 128`/`--img 112` nếu OOM. Internet OFF → thêm `--no-pretrained`.

In [ ]:
HF_FLAG=f'--hf-repo {HF_REPO}' if HF_REPO else ''
!python {CODE}/train.py --root {DATA} --index {DATA}/index.csv --classes {DATA}/classes.json \
    --out /kaggle/working/checkpoints --epochs 35 --batch 256 --img 128 --workers 2 \
    --resume {HF_FLAG}

## 5. Nghiệm thu — so trực tiếp DINOv2

| Test | DINOv2 | Đạt khi |
|---|---|---|
| T2 separation | +0.012 | **≥ +0.20** |
| T3 retrieval top-1 | 0.0% | **≥ 80%** |

In [ ]:
!python {CODE}/eval_discrim.py --root {DATA} --index {DATA}/index.csv \
    --ckpt /kaggle/working/checkpoints/best.pt

## 6. Checkpoint ở đâu — download & KHÔNG mất khi restart

Checkpoint: **`/kaggle/working/checkpoints/`** (`best.pt`, `last.pt`).

**Cách giữ + tải về (chọn 1):**
1. **Save & Run All (Commit)** *(đơn giản, đủ cho job ≤12h)*: chạy notebook ở chế độ commit → toàn bộ `/kaggle/working/` lưu thành **Output của Version** → tải từ tab Output; **sống sau khi đóng session**. Train 1.5–3h gọn trong 1 commit.
2. **HF Hub** *(bền + resume qua restart)*: set `HF_TOKEN` (Add-ons→Secrets) + `HF_REPO` ở cell 3 → push `last.pt`/`best.pt` lên HF **mỗi epoch**. Bị ngắt → chạy lại cell 4 (đã `--resume`) sẽ kéo `last.pt` từ HF và **tiếp từ epoch dở**. Tải từ HF bất cứ lúc nào.

⚠️ `/kaggle/working` bị XOÁ khi session **restart** nếu CHƯA commit → muốn an toàn tuyệt đối dùng cách 2 (HF).

In [ ]:
import os
ck='/kaggle/working/checkpoints'
print('checkpoints:', os.listdir(ck) if os.path.exists(ck) else 'chưa có (chạy cell 4)')
print('Tải về: Save Version -> tab Output, hoặc từ HF repo nếu bật cách 2.')